# Philadelphia, PA — Land Value Tax + Universal Basic Income## What this notebook modelsTax **100% of annual land rent**, leave the building tax exactly as billed today, hold the Cityand the School District harmless on the land portion of their current revenue, and pay theresidual out as an **equal per-capita dividend to every Philadelphia resident**.## Why this deviates from the standard 7-section templateEvery other city notebook in this repo models a revenue-neutral change to the *rate* or *base* ofthe property tax, and `model_wage_tax_swap.ipynb` swaps one tax instrument for another. This onedoes neither. It charges an **imputed rent flow** rather than an assessed stock, it is deliberately**not revenue-neutral** — raising more than today's levy is the whole point, since the surplus iswhat funds the dividend — and its primary result is a **transfer between people**, not aredistribution of a fixed bill between parcels. `save_standard_export` and `create_city_report`therefore do not apply; this notebook uses `lvt.ubi_utils` and `lvt.viz.create_lvt_ubi_report`.Methodology, derivations and the full limitations list: `docs/LVT_UBI_GUIDE.md`.## Policy assumptions| Choice | Setting || --- | --- || Land rent captured | 100% (`CAPTURE_RATE = 1.0`) || Building tax | unchanged, at today's combined 1.3998% || Revenue framing | City + School District held harmless; only the surplus is distributed || Currently-exempt land | stays exempt (rent basis = `taxable_land`); reported as a sensitivity || Capitalization rate | 5% net (`r − g`), swept over 3–7% || Dividend | equal per resident, adults and children alike |## The arithmeticWith `L` = taxable land value, `B` = taxable building value, `t` = the combined city + schoolrate, `i` = the net capitalization rate and `phi` = the capture rate:- Assessed land value is a market price that already capitalizes the land tax, so annual site rent  is `R = L * (i + t)` — **not** `L * i`. At `i = 5%` that gross-up is 28% more rent.- New bill `= phi * R + t * B`. The building term is byte-identical to today's.- Parcel tax change `= L * (phi*(i+t) - t)`, which at `phi = 1` collapses exactly to **`i * L`**.- Owner residual `= (i+t)*(L+B) - new_tax`, which at `phi = 1` is **`i * B`**: a normal return on  the structure and zero on the land. That identity is the proof the rent levy and the retained  building tax do not double-count.- Land wealth destroyed `= dividend pot / i`, exactly. The one-time capitalization loss to  landowners equals the present value of the dividend stream — two views of one transfer, never  to be added together.## The five limitations that matter most1. **OPA's land values carry a 20% default ratio on ~45% of improved parcels.** The entire pot is   `i × sum(taxable_land)`, and the gross-up multiplies that administrative artifact by ~4.6×.   Set `LAND_VALUE_SOURCE = 'lycd'` to re-run on the repo's independent land surface.2. **At 100% capture the levy cannot be a millage.** Land value capitalizes to zero, so the   ad-valorem base the levy would be denominated in ceases to exist and OPA would have to assess   rental values it does not publish today.3. **Assessment error becomes confiscation at full capture.** Effective capture is   `phi × (assessed / true)`; on vacant land roughly half of parcels exceed 100% of true rent, measured against sales (`docs/VACANT_LAND_VALUATION.md`). Retreating to 85% capture barely helps — `phi` scales every parcel equally while the problem is dispersion.4. **Tract results mix two incidence populations.** A tract's land rent is paid by owners who may   live elsewhere; its dividend goes to residents. Directional, not a household-level guarantee.5. **Static.** No behavioural response in land use, development, rents or migration — under a   reform whose entire purpose is to change them.

## Section 1 — Imports and Constants

In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.census_utils import get_census_tracts_shapefile, match_to_census_tracts
from lvt.lvt_utils import calculate_category_tax_summary, calculate_current_tax
from lvt.philadelphia import parcel_cache_path, tax_year_params, split_zero_building_parcels
from lvt.ubi_utils import (
    allocate_dividend_to_tracts,
    compute_breakeven_land_value,
    compute_ubi_dividend,
    get_population_by_tract,
    model_full_land_rent_tax,
    save_ubi_parcel_export,
    save_ubi_tract_export,
    summarize_ubi_incidence,
    sweep_land_rent_parameters,
)
from lvt.viz import create_lvt_ubi_report

STATE_FIPS = '42'
COUNTY_FIPS = '101'
FIPS = STATE_FIPS + COUNTY_FIPS

# --- Tax year ---------------------------------------------------------------------
# Rates and the revenue-validation target live in lvt/philadelphia.py, keyed by year and
# cited there. Do NOT hardcode 0.013998 here: the combined rate has been 1.3998% for years,
# but the City/School split moved at TY2025, which silently invalidates the city-only
# cross-check without changing anything the model computes.
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
COMBINED_RATE = TY.combined_rate_pct / 100             # fraction, e.g. 0.013998
MILLAGE = TY.combined_mills                            # per $1,000, e.g. 13.998

# --- Policy parameters ------------------------------------------------------------
DISCOUNT_RATE = 0.05          # net capitalization rate (r - g), NOT a raw discount rate
CAPTURE_RATE = 1.0            # share of land rent taken by the levy; 1.0 = full capture
REVENUE_FRAMING = 'city_held_harmless'   # or 'full_redistribution'
RENT_BASIS = 'taxable'        # 'taxable' (exemptions preserved) or 'full' (reaches exempt land)
LAND_VALUE_SOURCE = 'opa'     # 'opa' = OPA taxable_land; 'lycd' = the repo's LYCD land surface
CHILD_SHARE = 1.0             # every resident gets a full share, children included
ACS_YEAR = 2022

DISCOUNT_RATE_SWEEP = (0.03, 0.04, 0.05, 0.06, 0.07)
CAPTURE_RATE_SWEEP = (0.25, 0.50, 0.75, 0.85, 1.00)

# --- Derived naming ---------------------------------------------------------------
_SOURCE_SUFFIX = '' if LAND_VALUE_SOURCE == 'opa' else f'_{LAND_VALUE_SOURCE}'
CITY_NAME = f'philadelphia_lvt_ubi{_SOURCE_SUFFIX}_ty{TAX_YEAR}'
MODEL_TYPE = (f'lvt_ubi:capture_{CAPTURE_RATE:g},cap_rate_{DISCOUNT_RATE:g},'
              f'{REVENUE_FRAMING},rent_basis_{RENT_BASIS},land_{LAND_VALUE_SOURCE}')

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
PARCEL_PATH = parcel_cache_path(TAX_YEAR, DATA_DIR)

print(TY.describe())
print(f'Capitalization rate {DISCOUNT_RATE:.1%} | capture {CAPTURE_RATE:.0%} | '
      f'framing {REVENUE_FRAMING} | rent basis {RENT_BASIS} | land {LAND_VALUE_SOURCE}')
print(f'Export slug: {CITY_NAME}')

C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection) | homestead $100,000
Capitalization rate 5.0% | capture 100% | framing city_held_harmless | rent basis taxable | land opa
Export slug: philadelphia_lvt_ubi_ty2026


## Section 2 — Load Parcel Data (shared, read-only cache)This notebook only reads the shared year-keyed parcel cache; `model.ipynb` owns building it.

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'category_code'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
for _col in ('taxable_land', 'taxable_building', 'exempt_land', 'exempt_building', 'market_value'):
    gdf[_col] = pd.to_numeric(gdf[_col], errors='coerce').fillna(0.0)

print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base:  ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')
print(f'  taxable land:  ${gdf["taxable_land"].sum()/1e9:.3f}B')
print(f'  exempt land:   ${gdf["exempt_land"].sum()/1e9:.3f}B')

Loaded 583,249 parcels for TY2026
  taxable base:  $152.997B
  taxable land:  $43.000B
  exempt land:   $10.779B


## Section 3 — Land-Value Source

The dividend pot is `i × sum(land value)`, so it inherits whatever bias the land assessment
carries — this is the single largest source of uncertainty in the whole analysis, larger than the
capitalization rate. OPA gives roughly 45% of improved parcels a land ratio of exactly 0.200, its
default formula rather than a market observation, which biases the land base *downward*.

`LAND_VALUE_SOURCE = 'lycd'` re-runs the identical model on the repo's independent LYCD land
surface (`zone_psf × lot area`), whose land base is about 46% larger ($62.6B vs $43.0B at TY2026 —
not the ~26% a naive read of the two split-rate land millages suggests; see the guard comment in
the next cell). Most of that gap is vacant land, which LYCD values far above the assessor, so the
LYCD run is an upper bracket rather than a better estimate. It reads the already-exported
`analysis/data/philadelphia_lycd_ty<YEAR>.csv`, so run `model_lycd.ipynb` first.

**Only the rent basis is taken from that export. The baseline stays OPA.** Today's bill is rebuilt
from `taxable_land + taxable_building` under either source, because LYCD replaces land without
re-deriving buildings and so implies no parcel total anybody was billed on. Pairing LYCD land with
the OPA building overstated the modeled TY2026 levy by $274M (12.8%), and was caught by neither the
revenue validation in Section 5 nor the zero-sum check in Section 8 — see Limitation 19 in
`docs/LVT_UBI_GUIDE.md`. `baseline_total_col` in Section 6 is the guard against it.

Two further quirks of that export are avoided rather than depended on: its `property_category`
column is cp1252-mangled UTF-8 (a known upstream problem), and its row order does not match the
parcel cache. This notebook classifies from the cache's own `category_code` and joins on
`parcel_id` rather than by position.

In [3]:
if LAND_VALUE_SOURCE == 'opa':
    gdf['model_land'] = gdf['taxable_land']
    print(f'OPA taxable land base: ${gdf["model_land"].sum()/1e9:.3f}B')

elif LAND_VALUE_SOURCE == 'lycd':
    _lycd_path = REPO_ROOT / 'analysis' / 'data' / f'philadelphia_lycd_ty{TAX_YEAR}.csv'
    if not _lycd_path.exists():
        raise FileNotFoundError(
            f'{_lycd_path} not found. Run cities/philadelphia/model_lycd.ipynb for TY{TAX_YEAR} '
            'first, or set LAND_VALUE_SOURCE = "opa".'
        )
    _lycd = pd.read_csv(_lycd_path, usecols=['parcel_id', 'taxable_land_value'],
                        dtype={'parcel_id': str})
    # Ten parcel_ids genuinely appear twice in the Philadelphia exports with different
    # attributes; keep the first occurrence and say how many were dropped.
    _dupes = int(_lycd['parcel_id'].duplicated().sum())
    _lycd = _lycd.drop_duplicates(subset='parcel_id', keep='first')
    _lycd['_key'] = _lycd['parcel_id'].astype(str).str.strip().str.lstrip('0')

    gdf['_key'] = gdf['parcel_number'].str.lstrip('0')
    gdf = gdf.merge(_lycd[['_key', 'taxable_land_value']], on='_key', how='left')
    _matched = gdf['taxable_land_value'].notna().mean()
    gdf['model_land'] = gdf['taxable_land_value'].fillna(gdf['taxable_land'])
    gdf = gdf.drop(columns=['_key', 'taxable_land_value'])

    _ratio = gdf['model_land'].sum() / gdf['taxable_land'].sum()
    print(f'LYCD land joined to {_matched*100:.1f}% of parcels ({_dupes} duplicate ids dropped)')
    print(f'LYCD land base: ${gdf["model_land"].sum()/1e9:.3f}B  '
          f'({_ratio:.3f}x the OPA base of ${gdf["taxable_land"].sum()/1e9:.3f}B)')
    assert _matched > 0.95, f'only {_matched*100:.1f}% of parcels matched the LYCD export'
    # Band is on the LAND-BASE ratio L_lycd/L_opa: 1.456 observed at TY2026, 1.375 in the
    # original (TY2024) export. It is NOT the ratio of the two models' split-rate land
    # millages (29.001 OPA vs 23.076 LYCD = 1.257): revenue neutrality at 4:1 makes millage
    # vary as 4000*T/(4L+B), so that ratio is (4*L_lycd+B_lycd)/(4*L_opa+B_opa), a different
    # quantity. The earlier 1.15-1.40 band came from that mis-derivation and rejected a
    # correct LYCD run.
    assert 1.25 < _ratio < 1.65, (
        f'LYCD/OPA land base ratio {_ratio:.3f} is outside the expected 1.25-1.65 band '
        '— check the join key and the tax year.'
    )
else:
    raise ValueError(f"LAND_VALUE_SOURCE must be 'opa' or 'lycd', got {LAND_VALUE_SOURCE!r}")

OPA taxable land base: $43.000B


## Section 4 — Classify and Build the Rent BasisThe category map and its four stacked overrides are copied from `model.ipynb` — the who-pays chartneeds them, and the order matters (Override 3 must precede Override 4).**This notebook deliberately skips `model.ipynb`'s abated-parcel imputation**(`model_building = 4 × taxable_land`). That imputation exists so a revenue-neutral split-ratesolve does not see an artificially land-only parcel. Here the reform does not touch the improvementbase at all, so current law — a $0 building bill during the abatement — *is* the correctcounterfactual. Abated parcels consequently pay full land rent while their building stays free,which is a real interaction of the two policies rather than something to smooth over.

In [4]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce').astype('Int64').astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: a $0 taxable building line has three different causes, and calling all of
# them "abated" put ~13K homesteaded rowhomes in the abated bucket -- then revoked their
# Homestead Exemption under the reform. Split on the year's statutory homestead cap.
GENUINE_VACANT_CODES = {'6', '12', '13'}
_zb = split_zero_building_parcels(
    gdf, gdf['PROPERTY_CATEGORY'], TY.homestead_exemption, CATEGORY_MAP,
    genuine_vacant_codes=tuple(GENUINE_VACANT_CODES),
)
gdf['PROPERTY_CATEGORY'] = _zb.category
abated_mask = _zb.abated
print(_zb.describe())

# Override 3: OPA-vacant parcels that nonetheless carry a building record
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) & (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: re-classify fully exempt parcels by their OPA type
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code'].map(EXEMPT_CATEGORY_MAP).fillna('Other — Exempt')
)

# Full assessed land, for the RENT_BASIS = 'full' sensitivity in Section 9.
gdf['full_assessed_land'] = gdf['model_land'] + gdf['exempt_land']

# The rent basis is always named explicitly, never left to default to the baseline column.
# Under LAND_VALUE_SOURCE = 'opa' model_land is a copy of taxable_land, so this is the same
# surface under a different name; under 'lycd' it is a different surface entirely, and the
# baseline must not follow it there.
RENT_BASIS_COL = 'full_assessed_land' if RENT_BASIS == 'full' else 'model_land'

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print(gdf['PROPERTY_CATEGORY'].value_counts().head(12).to_string())

zero-building line: 14,287 abated | 13,995 homestead-zeroed (96.3% confirmed by OPA's homestead flag) | 1,119 genuinely $0 improvement


Total parcels: 583,249
Fully exempt: 36,932  |  Abated: 14,287  |  Improved vacant: 880  |  Taxable: 546,317

PROPERTY_CATEGORY
Single Family Residential             430570
Small Multi-Family (2-4 units)         38685
Vacant Land                            30557
Single Family Residential — Exempt     20078
Abated / Construction Exemption        14287
Mixed Use                              13743
Vacant Land — Exempt                   11714
Commercial                              8802
Industrial                              3553
Commercial — Exempt                     3298
Large Multi-Family (5+ units)           3002
Other Residential                       1108


## Section 5 — Baseline: Today's Property TaxSame current-tax reconstruction and city-only cross-check as `model.ipynb`, so the two notebooksstart from an identical baseline. The second assert is a wiring check against that notebook'spublished TY2026 figure — it catches a mis-joined or wrong-vintage cache before the reform mathruns on it.

In [5]:
gdf['millage_rate'] = MILLAGE
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_revenue = gdf['taxable_total'].mul(TY.city_mills / 1000).sum()
print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion ({TY.city_rate_pct}%):   ${city_revenue:,.0f}')

if TY.city_revenue_target is None:
    print(f'\nNO REVENUE VALIDATION for TY{TAX_YEAR}.')
    print(f'  {TY.source}')
else:
    gap_pct = (city_revenue / TY.city_revenue_target - 1) * 100
    print(f'City-only target ({TY.target_kind}):            ${TY.city_revenue_target:,}')
    print(f'City portion gap: {gap_pct:+.2f}%  (expected: a few % over, from delinquency)')
    assert abs(gap_pct) < 10.0, (
        f'City gap {gap_pct:.2f}% exceeds 10% for TY{TAX_YEAR}. Check that the assessment '
        f'year, the City rate ({TY.city_rate_pct}%) and the revenue target all refer to the '
        'same billing year — see lvt/philadelphia.py.'
    )

# Wiring check against model.ipynb's published TY2026 combined levy.
if TAX_YEAR == 2026:
    _reference = 2_142_649_826
    _drift = abs(current_revenue / _reference - 1)
    assert _drift < 0.02, (
        f'Combined levy ${current_revenue:,.0f} is {_drift*100:.1f}% away from '
        f"model.ipynb's published TY2026 figure ${_reference:,} — the parcel cache is "
        'probably a different vintage. Rebuild it with --year 2026 --force.'
    )
    print(f'\nWiring check vs. model.ipynb TY2026 levy: {_drift*100:+.2f}% drift. OK.')

Modeled combined levy (city + school):  $2,141,653,043
Implied city-only portion (0.6159%):   $942,308,979
City-only target (projection):            $891,102,000
City portion gap: +5.75%  (expected: a few % over, from delinquency)

Wiring check vs. model.ipynb TY2026 levy: +0.05% drift. OK.


## Section 6 — The Land Rent LevyFully-exempt parcels are held out of the levy exactly as they are held out of every otherPhiladelphia model, and every identity from the header is asserted here rather than described.

In [6]:
# land_value_col is the BASELINE -- the assessor's land, so current_tax reconstructs the
# real bill -- and rent_basis_col is the surface the rent is priced from. They are the same
# values under 'opa' and different ones under 'lycd'. baseline_total_col makes conflating
# them raise instead of quietly overstating every parcel's current bill.
summary, gdf = model_full_land_rent_tax(
    gdf,
    land_value_col='taxable_land',
    improvement_value_col='taxable_building',
    rent_basis_col=RENT_BASIS_COL,
    rent_basis_label=RENT_BASIS,
    baseline_total_col='taxable_total',
    discount_rate=DISCOUNT_RATE,
    combined_rate=COMBINED_RATE,
    capture_rate=CAPTURE_RATE,
    revenue_framing=REVENUE_FRAMING,
    exemption_flag_col='full_exmp',
    verbose=True,
)

Rent basis:                taxable (model_land)
Land value (rent basis):   $42,999,613,045
Current levy (baseline):   $2,141,653,043  (+0.0000% vs. taxable_total)
Imputed annual land rent:  $2,751,889,236
New land levy:             $2,751,889,236
Current land tax:          $601,908,583
Building tax (unchanged):  $1,539,744,459
Dividend pot (city_held_harmless): $2,149,980,652
Implied land millage equiv:  63.9980
Land wealth destroyed:     $42,999,613,045


In [7]:
_rent_land = gdf['rent_basis_land']
_base_land = gdf['taxable_land'].clip(lower=0).where(gdf['full_exmp'] != 1, 0.0)
_bldg = gdf['taxable_building'].clip(lower=0).where(gdf['full_exmp'] != 1, 0.0)

# The baseline is the bill, not a by-product of the rent surface. Asserted here as well as
# inside the model, because this is the invariant the LYCD run once broke silently.
assert np.allclose(gdf['current_tax'],
                   COMBINED_RATE * gdf['taxable_total'].where(gdf['full_exmp'] != 1, 0.0),
                   rtol=1e-9), 'current_tax must be the actual levy on the OPA assessment'
assert abs(summary['total_current_tax'] / current_revenue - 1) < 1e-9, \
    "the reform's baseline and Section 5's rebuilt levy must be the same number"

# Identity 1 — i * L only when the rent is priced off the land the current tax is billed on.
# Under 'lycd', or a rent basis reaching exempt land, the general form applies instead.
_bases_match = summary['rent_basis_matches_current_basis']
if CAPTURE_RATE == 1.0:
    if _bases_match:
        assert np.allclose(gdf['tax_change'], DISCOUNT_RATE * _rent_land, rtol=1e-9), \
            'tax_change should equal discount_rate * land at full capture on a matched basis'
    else:
        assert np.allclose(
            gdf['tax_change'],
            (DISCOUNT_RATE + COMBINED_RATE) * _rent_land - COMBINED_RATE * _base_land,
            rtol=1e-9), 'tax_change should be phi*(i+t)*L_rent - t*L_assessed'

    # Identity 2 — the owner keeps a normal return on the structure and nothing on the land.
    # No precondition: it never touches the baseline.
    assert np.allclose(gdf['owner_residual'], DISCOUNT_RATE * _bldg, rtol=1e-9), \
        'owner_residual should equal discount_rate * building at full capture'

# Identity 3 — the stock destroyed is the present value of the flow.
assert np.isclose(summary['land_wealth_destroyed'], summary['ubi_pot'] / DISCOUNT_RATE,
                  rtol=1e-9), 'wealth closure broken'

# The building tax is untouched.
assert np.allclose(gdf['building_tax'], COMBINED_RATE * _bldg, rtol=1e-9), \
    'building_tax must be exactly today\'s ad-valorem bill on improvements'

# The two framings differ by exactly today's land-tax revenue.
assert np.isclose(summary['full_redistribution_pot'] - summary['city_held_harmless_pot'],
                  summary['total_current_land_tax'], rtol=1e-9)

print(f"Baseline levy (= Section 5):  ${summary['total_current_tax']/1e9:.3f}B  "
      f"({summary['baseline_drift_pct']:+.4f}% vs. t x taxable_total)")
print(f"Held-harmless land credit:    ${summary['total_current_land_tax']/1e9:.3f}B  "
      f"(the assessor's land revenue{'' if _bases_match else ', not the rent basis'})")
print(f"Dividend pot:                 ${summary['ubi_pot']/1e9:.3f}B")
print(f"  (full-redistribution pot:   ${summary['full_redistribution_pot']/1e9:.3f}B, "
      f"leaving a ${summary['full_redistribution_budget_hole']/1e6:,.0f}M budget hole)")
print(f"Land wealth destroyed:        ${summary['land_wealth_destroyed']/1e9:.1f}B one-time")
print(f"Implied land millage equiv:   {summary['implied_land_millage_equivalent']:.4f} "
      f"(today's combined rate: {MILLAGE:.3f})")

Baseline levy (= Section 5):  $2.142B  (+0.0000% vs. t x taxable_total)
Held-harmless land credit:    $0.602B  (the assessor's land revenue)
Dividend pot:                 $2.150B
  (full-redistribution pot:   $2.752B, leaving a $602M budget hole)
Land wealth destroyed:        $43.0B one-time
Implied land millage equiv:   63.9980 (today's combined rate: 13.998)


In [8]:
# Magnitude gates. These are the acceptance test for the first real run: the reform's
# arithmetic is proven by the unit tests in tests/test_ubi_utils.py, so anything failing
# here is a data-wiring problem, not a formula problem.
if TAX_YEAR == 2026 and CAPTURE_RATE == 1.0 and RENT_BASIS == 'taxable':
    _land_total = summary['total_land_value_rent_basis']
    _expected_band = (34e9, 48e9) if LAND_VALUE_SOURCE == 'opa' else (52e9, 74e9)
    assert _expected_band[0] < _land_total < _expected_band[1], (
        f'Land base ${_land_total/1e9:.1f}B is outside the expected '
        f'${_expected_band[0]/1e9:.0f}-{_expected_band[1]/1e9:.0f}B band for '
        f'{LAND_VALUE_SOURCE.upper()} land at TY2026.'
    )
    assert np.isclose(summary['implied_land_millage_equivalent'],
                      (DISCOUNT_RATE + COMBINED_RATE) * 1000, rtol=1e-9)
    print(f'Magnitude gates passed: land base ${_land_total/1e9:.1f}B, '
          f'{summary["implied_land_millage_equivalent"]:.1f} implied mills.')

Magnitude gates passed: land base $43.0B, 64.0 implied mills.


## Section 7 — Population, Dividend, and the Tract JoinThe dividend denominator is ACS resident population summed over the same tracts every other numberhere is keyed to — never a hardcoded citywide figure, which would silently drift from the tracttable and break the adding-up check below.

In [9]:
pop_df = get_population_by_tract(FIPS, year=ACS_YEAR)
tract_boundaries = get_census_tracts_shapefile(FIPS)

tract_gdf = tract_boundaries.merge(pop_df, on='tract_geoid', how='left', suffixes=('', '_acs'))
tract_gdf = tract_gdf.loc[:, ~tract_gdf.columns.str.endswith('_acs')]
tract_gdf = gpd.GeoDataFrame(tract_gdf, geometry='geometry',
                             crs=tract_boundaries.crs or 'EPSG:4326')

TOTAL_POPULATION = float(tract_gdf['total_pop'].sum())
print(f'{len(tract_gdf):,} tracts, {tract_gdf["total_pop"].notna().sum():,} with ACS population')
print(f'Total population (ACS {ACS_YEAR} 5-year): {TOTAL_POPULATION:,.0f}')
print(f'Renter share of occupied households: '
      f'{tract_gdf["renter_households"].sum()/tract_gdf["occupied_households"].sum()*100:.1f}%')

408 tracts, 408 with ACS population
Total population (ACS 2022 5-year): 1,593,208
Renter share of occupied households: 47.8%


In [10]:
dividend = compute_ubi_dividend(summary['ubi_pot'], TOTAL_POPULATION, child_share=CHILD_SHARE)
UBI_PER_CAPITA = dividend['ubi_per_capita']

print(f"Dividend pot:        ${dividend['pot_total']/1e9:.3f}B")
print(f"Population:          {dividend['population']:,.0f}")
print(f"Dividend per resident: ${UBI_PER_CAPITA:,.0f}/yr")
print(f"  a 2.4-person household receives ${UBI_PER_CAPITA*2.4:,.0f}/yr")

if TAX_YEAR == 2026 and CAPTURE_RATE == 1.0 and LAND_VALUE_SOURCE == 'opa' \
        and RENT_BASIS == 'taxable' and DISCOUNT_RATE == 0.05:
    assert 900 < UBI_PER_CAPITA < 1_800, (
        f'Dividend ${UBI_PER_CAPITA:,.0f}/resident is outside the expected $900-1,800 band '
        'for OPA land at TY2026 and a 5% capitalization rate. Check the land base and the '
        'population denominator.'
    )

Dividend pot:        $2.150B
Population:          1,593,208
Dividend per resident: $1,349/yr
  a 2.4-person household receives $3,239/yr


In [11]:
# fallback_nearest: tract polygons stop at the water's edge, so a few river-front parcel
# centroids fall inside no tract. They must not be left unmatched -- the groupby below keeps
# a NaN-keyed group that the merge onto tract_gdf then silently discards, so those parcels'
# levy would vanish from the roll-up and break the adding-up identity asserted at the bottom
# of this cell. Assigning them to the nearest tract keeps every dollar accounted for.
gdf = match_to_census_tracts(gdf, tract_gdf[['tract_geoid', 'geometry']],
                             fallback_nearest=True)
_unmatched = int(gdf['tract_geoid'].isna().sum())
# Printed at 4 decimals on purpose: at 2 decimals, 3 unmatched parcels in 583k round to
# '100.00%' and the roll-up shortfall looks like it came from nowhere.
print(f'Parcel -> tract join coverage: '
      f'{gdf["tract_geoid"].notna().mean()*100:.4f}%  ({_unmatched:,} unmatched)')
assert _unmatched == 0, (
    f'{_unmatched:,} parcels landed in no tract; their levy would be dropped by the '
    'roll-up merge below. Expected 0 with fallback_nearest=True.'
)

_rolled = (gdf.groupby('tract_geoid', dropna=False)[
    ['new_land_tax', 'current_tax', 'new_tax', 'tax_change', 'model_land']]
    .sum().reset_index())
tract_gdf = tract_gdf.merge(_rolled, on='tract_geoid', how='left')
for _col in ('new_land_tax', 'current_tax', 'new_tax', 'tax_change', 'model_land'):
    tract_gdf[_col] = tract_gdf[_col].fillna(0.0)

tract_gdf = allocate_dividend_to_tracts(tract_gdf, UBI_PER_CAPITA,
                                        pot_total=summary['ubi_pot'])

# Adding-up: the levy and the dividend must both survive the roll-up intact.
assert np.isclose(tract_gdf['new_land_tax'].sum(), summary['total_new_land_tax'], rtol=1e-6), \
    'land levy did not survive the tract roll-up'
assert np.isclose(tract_gdf['tract_dividend'].sum(), summary['ubi_pot'], rtol=1e-6), \
    'allocated dividends do not sum to the pot'
print('Roll-up adding-up checks passed.')

Parcel -> tract join coverage: 100.0000%  (0 unmatched)
Roll-up adding-up checks passed.


## Section 8 — Winners and Losers`net_gain = dividend received − additional tax paid`. **Positive means the tract's residents comeout ahead** — the opposite of the sign convention in the wage-tax swap export, where positive meanspaying more.The subtrahend is `tax_change`, the levy *above today's bill*, not the whole land levy: under theheld-harmless framing the taxing bodies keep the portion the land already pays, so counting itagainst residents would double-count it. That also makes the reform exactly zero-sum acrosstracts — which the assert below states rather than assumes.

In [12]:
tract_export = save_ubi_tract_export(
    tract_gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    discount_rate=DISCOUNT_RATE,
    capture_rate=CAPTURE_RATE,
    ubi_per_capita=UBI_PER_CAPITA,
)
for _col in ('net_gain', 'net_gain_per_capita', 'net_gain_pct'):
    tract_gdf[_col] = tract_export[_col].values

if REVENUE_FRAMING == 'city_held_harmless':
    assert abs(tract_gdf['net_gain'].sum()) < 1.0, (
        'held harmless, dividends and additional tax must net to zero citywide; '
        f"got ${tract_gdf['net_gain'].sum():,.0f}"
    )
    print('Zero-sum check passed: every dollar of dividend is a dollar of additional land tax.')

_winners = tract_gdf['net_gain'] > 0
print(f'\nTracts where residents come out ahead: {_winners.sum():,} of {len(tract_gdf):,}')
print(f'Residents living in those tracts: '
      f'{tract_gdf.loc[_winners, "total_pop"].sum()/TOTAL_POPULATION*100:.1f}%')
print(f'Median net gain per resident: ${tract_gdf["net_gain_per_capita"].median():,.0f}/yr')

Zero-sum check passed: every dollar of dividend is a dollar of additional land tax.

Tracts where residents come out ahead: 236 of 408
Residents living in those tracts: 67.5%
Median net gain per resident: $267/yr


In [13]:
BREAKEVEN_HOUSEHOLD_SIZE = float(
    np.average(tract_gdf['avg_household_size'].fillna(0),
               weights=tract_gdf['occupied_households'].fillna(0))
)
BREAKEVEN_LAND_VALUE = compute_breakeven_land_value(
    UBI_PER_CAPITA, BREAKEVEN_HOUSEHOLD_SIZE,
    discount_rate=DISCOUNT_RATE, combined_rate=COMBINED_RATE, capture_rate=CAPTURE_RATE,
)

# Scoped to single-unit residential on purpose: this compares a parcel-level bill against a
# household-level dividend, which is only meaningful where one parcel houses one household.
_sfr = gdf[(gdf['category_code'] == '1') & (gdf['full_exmp'] == 0)]

# A single threshold in L presumes identity 1. Comparing tax_change against the household
# dividend directly is exact under either land source and reduces to the same parcels when
# the bases match -- which is why the two numbers below are equal on an OPA run.
_household_dividend = UBI_PER_CAPITA * BREAKEVEN_HOUSEHOLD_SIZE
_below = float((_sfr['tax_change'] < _household_dividend).mean() * 100)
_below_threshold = float((_sfr['rent_basis_land'] < BREAKEVEN_LAND_VALUE).mean() * 100)

print(f'Household size (occupancy-weighted): {BREAKEVEN_HOUSEHOLD_SIZE:.2f}')
print(f'Break-even land value: ${BREAKEVEN_LAND_VALUE:,.0f}')
print(f'Single-family parcels ahead of the dividend: {_below:.1f}% of {len(_sfr):,}')
print(f'  median SFR land value (rent basis): ${_sfr["rent_basis_land"].median():,.0f}')
if abs(_below - _below_threshold) > 0.05:
    print(f'  breakeven_curve.png draws the closed-form threshold instead: '
          f'{_below_threshold:.1f}% — indicative only, the rent basis is not the billed land')

Household size (occupancy-weighted): 2.34
Break-even land value: $63,278
Single-family parcels ahead of the dividend: 76.5% of 442,799
  median SFR land value (rent basis): $43,000


In [14]:
incidence = summarize_ubi_incidence(gdf, tract_gdf, ubi_per_capita=UBI_PER_CAPITA)

print('Annual land rent collected, by payer class:')
_total_rent = sum(incidence['rent_by_payer_class'].values())
for _cls, _amt in incidence['rent_by_payer_class'].items():
    print(f'  {_cls:<28} ${_amt/1e9:6.3f}B  ({_amt/_total_rent*100:5.1f}%)')
print()
print(f"Renter households: {incidence['renter_households']:,.0f} "
      f"({incidence['renter_share_pct']:.1f}% of occupied units) — they collect the dividend "
      'and, because a tax on pure land rent cannot be shifted to tenants, owe none of the levy.')
print(f"Residents in net-positive tracts: {incidence['share_population_net_positive']:.1f}%")

Annual land rent collected, by payer class:
  Residential (1-4 units)      $ 1.661B  ( 60.4%)
  Commercial / Industrial      $ 0.546B  ( 19.8%)
  Vacant land / Parking        $ 0.241B  (  8.8%)
  Abated construction          $ 0.168B  (  6.1%)
  Multi-family (5+ units)      $ 0.135B  (  4.9%)
  Other / Institutional        $ 0.001B  (  0.1%)

Renter households: 314,980 (47.8% of occupied units) — they collect the dividend and, because a tax on pure land rent cannot be shifted to tenants, owe none of the levy.
Residents in net-positive tracts: 67.5%


In [15]:
category_summary = calculate_category_tax_summary(
    df=gdf[gdf['full_exmp'] == 0],
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print(category_summary[['PROPERTY_CATEGORY', 'property_count', 'median_tax_change',
                        'median_tax_change_pct', 'total_tax_change_dollars']].to_string(index=False))

              PROPERTY_CATEGORY  property_count  median_tax_change  median_tax_change_pct  total_tax_change_dollars
      Single Family Residential          430570           2128.000              84.904591              1.083289e+09
 Small Multi-Family (2-4 units)           38685           3287.000              71.438777              2.139540e+08
                    Vacant Land           30557           1705.000             357.193885              1.867478e+08
Abated / Construction Exemption           14287           5060.000             357.193885              1.314250e+08
                      Mixed Use           13743           1858.000              71.438777              4.060193e+07
                     Commercial            8802           3704.500              71.438777              1.589476e+08
                     Industrial            3553           5482.500             107.158165              9.519535e+07
  Large Multi-Family (5+ units)            3002          11478.750      

## Section 9 — Sensitivity

Two sweeps and one re-run. The important result is the third cell: **the set of tracts whose
residents come out ahead barely moves with the capitalization rate.** The rate scales every dollar
figure in this notebook.

Where the rent is priced off the land the current tax is billed on — the `'opa'` default — the
partition is *exactly* invariant to it, and by the same algebra to any uniform bias in the level of
assessment. Only *non-uniform* assessment error redistributes.

Under `'lycd'` the rent surface is not the billed one, so a tract's net position is
`(i+t)·(ΣL_rent·s_j − L_rent_j) − t·(ΣL_opa·s_j − L_opa_j)` and raising `i` re-weights the two
surfaces against each other. Each tract can cross the line at most once, at
`i* = t(B_j − A_j)/A_j`. Because `t ≪ i` the effect is small — a handful of tracts and well under a
point of population across 3–9% — but it is not zero, and the cell measures it rather than
asserting it away.

In [16]:
sweep = sweep_land_rent_parameters(
    gdf, 'taxable_land', 'taxable_building', population=TOTAL_POPULATION,
    discount_rates=DISCOUNT_RATE_SWEEP, capture_rates=CAPTURE_RATE_SWEEP,
    base_discount_rate=DISCOUNT_RATE, base_capture_rate=CAPTURE_RATE,
    rent_basis_col=RENT_BASIS_COL, baseline_total_col='taxable_total',
    combined_rate=COMBINED_RATE,
    revenue_framing=REVENUE_FRAMING, exemption_flag_col='full_exmp',
)
_show = sweep.copy()
_show['ubi_pot'] = (_show['ubi_pot'] / 1e9).round(3)
_show['ubi_per_capita'] = _show['ubi_per_capita'].round(0)
_show['land_wealth_destroyed'] = (_show['land_wealth_destroyed'] / 1e9).round(1)
print(_show.rename(columns={'ubi_pot': 'ubi_pot_$B',
                            'land_wealth_destroyed': 'wealth_destroyed_$B'}).to_string(index=False))

        swept  discount_rate  capture_rate  total_land_rent  ubi_pot_$B  ubi_per_capita  implied_land_millage_equivalent  wealth_destroyed_$B
discount_rate           0.03          1.00     1.891897e+09       1.290           810.0                          43.9980                 43.0
discount_rate           0.04          1.00     2.321893e+09       1.720          1080.0                          53.9980                 43.0
discount_rate           0.05          1.00     2.751889e+09       2.150          1349.0                          63.9980                 43.0
discount_rate           0.06          1.00     3.181885e+09       2.580          1619.0                          73.9980                 43.0
discount_rate           0.07          1.00     3.611881e+09       3.010          1889.0                          83.9980                 43.0
 capture_rate           0.05          0.25     2.751889e+09       0.086            54.0                          15.9995                  1.7
 captu

In [17]:
# Reaching currently-exempt land is a policy choice, not a fact. Its size, for reference.
_exempt_summary, _ = model_full_land_rent_tax(
    gdf, 'taxable_land', 'taxable_building', rent_basis_col='full_assessed_land',
    rent_basis_label='full', baseline_total_col='taxable_total',
    discount_rate=DISCOUNT_RATE, combined_rate=COMBINED_RATE, capture_rate=CAPTURE_RATE,
    revenue_framing=REVENUE_FRAMING,
)
_exempt_ubi = _exempt_summary['ubi_pot'] / TOTAL_POPULATION
print(f'Rent basis "taxable" (modeled): ${UBI_PER_CAPITA:,.0f}/resident')
print(f'Rent basis "full"   (reaching exempt land): ${_exempt_ubi:,.0f}/resident '
      f'({_exempt_ubi/UBI_PER_CAPITA - 1:+.0%})')
print('Charging rent on institutional and government land would require changing '
      'Pennsylvania exemption law, and is fiscally circular for City- and School-owned parcels.')

Rent basis "taxable" (modeled): $1,349/resident
Rent basis "full"   (reaching exempt land): $1,782/resident (+32%)
Charging rent on institutional and government land would require changing Pennsylvania exemption law, and is fiscally circular for City- and School-owned parcels.


In [18]:
# The invariance result, demonstrated rather than asserted in prose -- along with its
# precondition, which is identity 1's. Priced off the land the current tax is billed on, a
# tract's net position is i * (sum(L)*s_j - L_j) and its sign does not involve i. Priced off
# a surface that is NOT the billed land it becomes
#
#     net_j = (i+t) * (sum(L_rent)*s_j - L_rent_j)  -  t * (sum(L_opa)*s_j - L_opa_j)
#
# -- two surfaces weighted (i+t) and t, so raising i re-weights them and a tract crosses at
# most once, at i* = t*(B_j - A_j)/A_j. t << i keeps that small; it does not make it zero.
_sets = {}
for _rate in (0.03, DISCOUNT_RATE, 0.09):
    _s, _p = model_full_land_rent_tax(
        gdf, 'taxable_land', 'taxable_building', rent_basis_col=RENT_BASIS_COL,
        baseline_total_col='taxable_total',
        discount_rate=_rate, combined_rate=COMBINED_RATE, capture_rate=CAPTURE_RATE,
        revenue_framing=REVENUE_FRAMING, exemption_flag_col='full_exmp')
    _t = (_p.groupby('tract_geoid')['tax_change'].sum()
          .reindex(tract_gdf['tract_geoid']).fillna(0.0).values)
    _d = tract_gdf['total_pop'].fillna(0).values * (_s['ubi_pot'] / TOTAL_POPULATION)
    _sets[_rate] = frozenset(tract_gdf['tract_geoid'][(_d - _t) > 0])

_rates = list(_sets)
_pop = tract_gdf.set_index('tract_geoid')['total_pop'].fillna(0.0)
_shares = {_r: _pop.reindex(list(_sets[_r])).sum() / TOTAL_POPULATION * 100 for _r in _rates}

if _bases_match:
    assert _sets[_rates[0]] == _sets[_rates[1]] == _sets[_rates[2]], \
        'on a matched basis the partition cannot depend on the capitalization rate'
    print(f'Net-positive tract set identical at {_rates[0]:.0%}, {_rates[1]:.0%} and '
          f'{_rates[2]:.0%}: {len(_sets[_rates[0]]):,} of {len(tract_gdf):,} tracts.')
    print('The capitalization rate scales the dividend; it does not change who wins.')
else:
    _movers = set().union(*[_sets[_r] ^ _sets[_rates[1]] for _r in _rates])
    print(f'Rent basis is not the billed land, so the partition is near-invariant rather than '
          f'invariant: {len(_movers)} of {len(tract_gdf):,} tracts cross the line anywhere '
          f'between {_rates[0]:.0%} and {_rates[-1]:.0%}.')
    for _r in _rates:
        print(f'  i={_r:.0%}: {len(_sets[_r]):,} net-positive tracts, '
              f'{_shares[_r]:.2f}% of residents')
    assert max(_shares.values()) - min(_shares.values()) < 2.0, (
        'the resident share coming out ahead moves more than 2 points across the '
        f'{_rates[0]:.0%}-{_rates[-1]:.0%} band; the headline conclusion is rate-dependent '
        'and cannot be reported without one'
    )
    print('The capitalization rate scales the dividend; it barely moves who wins.')

Net-positive tract set identical at 3%, 5% and 9%: 236 of 408 tracts.
The capitalization rate scales the dividend; it does not change who wins.


## Section 10 — Export and Report

In [19]:
parcel_export = save_ubi_parcel_export(
    gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}_parcels.csv',
    parcel_id_col='parcel_number',
    # The assessor's land, so taxable_land_value and current_tax agree row by row. The land
    # the rent was priced from rides along as rent_basis_land; on an OPA run they are equal.
    land_value_col='taxable_land',
    model_type=MODEL_TYPE,
    discount_rate=DISCOUNT_RATE,
    capture_rate=CAPTURE_RATE,
)

report = create_lvt_ubi_report(
    tract_gdf,
    CITY_NAME,
    output_dir='../../analysis/reports',
    show=False,
    parcels_df=gdf,
    land_value_col='model_land',
    ubi_per_capita=UBI_PER_CAPITA,
    incidence_summary=incidence,
    breakeven_land_value=BREAKEVEN_LAND_VALUE,
    sweep_df=sweep,
)

print(f'Tract export:  ../../analysis/data/{CITY_NAME}.csv  ({len(tract_export):,} tracts)')
print(f'Parcel export: ../../analysis/data/{CITY_NAME}_parcels.csv  ({len(parcel_export):,} parcels)')
print(f'Charts: {len(report["charts_saved"])} PNGs in analysis/reports/{CITY_NAME}/')
for _p in report['charts_saved']:
    print(f'  {Path(_p).name}')

Tract export:  ../../analysis/data/philadelphia_lvt_ubi_ty2026.csv  (408 tracts)
Parcel export: ../../analysis/data/philadelphia_lvt_ubi_ty2026_parcels.csv  (583,249 parcels)
Charts: 7 PNGs in analysis/reports/philadelphia_lvt_ubi_ty2026/
  net_per_capita_map.png
  income_quintile.png
  minority_quintile.png
  breakeven_curve.png
  who_pays.png
  discount_rate_sensitivity.png
  distribution.png


## Summary**The headline.** A 100% tax on Philadelphia land rent, with the building tax untouched and theCity and School District held whole, funds an annual dividend to every resident. Because a tax onland rent cannot be shifted to tenants, renters — roughly half of Philadelphia's occupiedhouseholds — collect the dividend and owe none of the levy. Owner-occupiers come out aheadwhenever their land value sits below the break-even threshold, which most single-family parcels do.The residents who lose are concentrated in the land-richest tracts, and a large share of the levyis collected from commercial, institutional and absentee owners who receive no dividend at all.That mismatch between who pays and who collects is the entire distributional result — the mirrorimage of the commuter transfer in the wage-tax swap, running the other way.**Stock and flow are two views of one transfer, and must never be added.** Current landowners beara one-time capitalization loss roughly equal to the whole assessed land base; residents receive anannual dividend whose present value is exactly that same figure. Anyone who buys land after thereform pays the annual levy but paid nothing for the land, and bears no net burden at all.**The strongest structural objection is not distributional, it is administrative.** At full capturethe market price of land goes to zero, so the ad-valorem base this levy would be denominated inceases to exist: sale prices stop being informative about land, land-secured mortgage collateralgoes to zero, and the assessor would have to publish imputed *rental* values that OPA does notproduce today. Assessment error also stops being a fairness problem and becomes confiscation —effective capture is `phi × (assessed / true)`, so any parcel over-assessed relative to its truerent is taxed at more than 100% of it. That, not the politics, is the argument for stopping near85% rather than 100%.**What the result does and does not depend on.** The capitalization rate and any uniform bias inthe level of assessment scale every dollar figure here, but the demonstration in Section 9 showsthey cannot change which tracts come out ahead. What *can* change it is non-uniform assessmenterror — and OPA's default 20% land ratio on roughly 45% of improved parcels is exactly that. Run`LAND_VALUE_SOURCE = 'lycd'` before treating any single number here as settled.Full methodology and the complete limitations list: `docs/LVT_UBI_GUIDE.md`.